# Conditional Chain

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnableLambda, RunnableBranch
from dotenv import load_dotenv
import os
from typing import Literal

c:\Users\arunk\anaconda3\envs\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [30]:
from pydantic import BaseModel, Field

In [31]:
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser

In [32]:
from langchain_groq import ChatGroq

In [33]:
load_dotenv()

True

In [34]:
class Sentiment(BaseModel):
    sentiment: Literal['Positive', 'Negative'] = Field(description='Generate the sentiment either Positive or Negative')

In [35]:
parser = PydanticOutputParser(pydantic_object=Sentiment)

In [36]:
prompt1 = PromptTemplate(template='Generate the sentiment of the following {feedback} in {format}', input_variables=['feedback'], partial_variables={'format':parser.get_format_instructions()})

In [37]:
gemini = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

In [38]:
gpt = ChatOpenAI(model='gpt-3.5-turbo')

In [39]:
groq = ChatGroq(model='llama-3.3-70b-versatile', api_key= os.getenv("GROQ_API_KEY"))

In [40]:
chainS = prompt1 | groq | parser

In [41]:
response = chainS.invoke({'feedback':'i Didnt liked this product'})

In [42]:
response

Sentiment(sentiment='Negative')

In [43]:
response = chainS.invoke({'feedback':'i Didnt liked this product'}).sentiment
response

'Negative'

In [44]:
response = chainS.invoke({'feedback':'i liked this product'}).sentiment

In [45]:
response

'Positive'

In [46]:
positive_prompt = PromptTemplate(template='Generate teh Positive Response for the Recieved {Sentiment} to the User', input_variables=['Sentiment'])

In [47]:
negative_prompt = PromptTemplate(template='Generate response as the Sorry Message for the Recieved {Sentiment} to the User', input_variables=['Sentiment'])

In [48]:
gpt = ChatOpenAI(model = 'gpt-3.5-turbo')

In [49]:
strParser = StrOutputParser()

In [50]:
conditional_chain = RunnableBranch(
    (
        lambda x: x.sentiment == 'Positive', positive_prompt | gpt |  strParser
    ),
    (
        lambda x: x.sentiment == 'Negative', negative_prompt | gpt |  strParser
    ),
    RunnableLambda(lambda x: 'Could not extract the Sentiment')
)

In [51]:
final_chain = chainS | conditional_chain 

In [53]:
result = final_chain.invoke({
    'Iphone 17 is an hype as I didnt liked it at all'
})

In [54]:
result

"I'm sorry to hear that you're feeling negative. Is there anything specific that's causing you to feel this way? Please know that I'm here to support you in any way I can."

In [55]:
result = final_chain.invoke({
    'I like the Iphone'
})

In [56]:
result

"That's great to hear! I'm glad you're feeling positive. Keep up the good vibes!"

In [57]:
final_chain.get_graph().print_ascii()

    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
      +----------+       
      | ChatGroq |       
      +----------+       
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
       +--------+        
       | Branch |        
       +--------+        
            *            
            *            
            *            
    +--------------+     
    | BranchOutput |     
    +--------------+     
